# Explore water detection

Run cells in order with **Shift+Enter**. Predict, run, inspect and explain. This activity uses local synthetic data and the repository application code. Follow `exercise.md` to launch the Python environment.

Setup writes outputs to a fresh directory under `.notebook-runs/`. Restart the kernel and run all cells to reset the activity. No satellite download is required.

In [ ]:
import os
import sys
from pathlib import Path
from tempfile import mkdtemp

import matplotlib.pyplot as plt
import numpy as np
import rasterio
from skimage.filters import threshold_otsu
from waterbodies import crop_impl, norm_diff_impl, otsu_impl, stac_impl
from waterbodies.common import read_item
from waterbodies.validation import validate_item

repository = next(
    directory
    for directory in (Path.cwd(), *Path.cwd().parents)
    if (directory / "scripts/fixtures.py").is_file()
)
if str(repository) not in sys.path:
    sys.path.insert(0, str(repository))

In [ ]:
from scripts.fixtures import create_fixture

# Keep files within the checkout so restarting the kernel can find the repository.
workspace = repository / ".notebook-runs"
workspace.mkdir(exist_ok=True)
run_directory = Path(mkdtemp(prefix="waterbodies-", dir=workspace))
source_catalog = run_directory / "source"
create_fixture(source_catalog)
os.chdir(run_directory)
print(f"Outputs: {run_directory}")

## 1. Inspect the green and NIR bands

Which half would you expect to contain water? Compare the colours and printed values. These 8×8 rasters use a deliberately simplified one-degree grid, not a realistic satellite resolution.

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(9, 4), constrained_layout=True)
for axis, band in zip(axes, ("green", "nir"), strict=True):
    with rasterio.open(source_catalog / f"{band}.tif") as raster:
        pixels = raster.read(1)
    image = axis.imshow(pixels, vmin=0, vmax=400, cmap="viridis")
    axis.set(title=band, xlabel="Column", ylabel="Row")
    print(f"{band}: left={pixels[0, 0]}, right={pixels[0, -1]}")
figure.colorbar(image, ax=axes, label="Synthetic band value")
plt.show()

## 2. Predict before running

Calculate `NDWI = (green - nir) / (green + nir)` for each half. The AOI is **west, south, east, north**, in EPSG:4326. How many rows and columns survive `1,1,7,7`? How many water pixels do you expect?

Replace `None` with your predictions. Blank predictions also allow the notebook to run.

In [ ]:
predicted_left_ndwi: float | None = None
predicted_right_ndwi: float | None = None
predicted_water_pixels: int | None = None

aoi = "1,1,7,7"
bands = ["green", "nir"]

## 3. Crop the bands

Inspect the cropped images, dimensions, bounds and CRS. Both bands must share the same grid.

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(9, 4), constrained_layout=True)
for axis, band in zip(axes, bands, strict=True):
    crop_impl.execute(input_item=source_catalog, aoi=aoi, epsg="4326", band=band)
    with rasterio.open(f"crop_{band}.tif") as raster:
        pixels = raster.read(1)
        print(f"{band}: shape={raster.shape}, bounds={tuple(raster.bounds)}, CRS={raster.crs}")
    axis.imshow(pixels, vmin=0, vmax=400, cmap="viridis")
    axis.set(title=f"Cropped {band}", xlabel="Column", ylabel="Row")
plt.show()

## 4. Calculate NDWI

The application expects **green, NIR** in that order. Compare the result with your predictions. Keep the colour scale fixed at −1 to +1 when experimenting.

In [ ]:
norm_diff_impl.execute(rasters=[Path(f"crop_{band}.tif") for band in bands])
with rasterio.open("norm_diff.tif") as raster:
    ndwi = raster.read(1)
figure, axis = plt.subplots(figsize=(5, 4), constrained_layout=True)
image = axis.imshow(ndwi, vmin=-1, vmax=1, cmap="BrBG")
axis.set(title="NDWI", xlabel="Column", ylabel="Row")
figure.colorbar(image, ax=axis, label="NDWI")
plt.show()
print(f"Observed NDWI values: {np.unique(ndwi)}")
print(f"Your predictions: left={predicted_left_ndwi}, right={predicted_right_ndwi}")

## 5. Classify and count water

Otsu selects a threshold from finite NDWI values. Water means **strictly greater than** the threshold. Nonfinite values become zero, so zero can mean non-water or an invalid calculation.

The histogram displays the threshold from the same Otsu library used by the application; the mask is produced by the application itself.

In [ ]:
otsu_impl.execute(raster=Path("norm_diff.tif"))
with rasterio.open("otsu.tif") as raster:
    water_mask = raster.read(1)
finite_ndwi = ndwi[np.isfinite(ndwi)]
cutoff = float(threshold_otsu(finite_ndwi))
figure, axes = plt.subplots(1, 2, figsize=(10, 4), constrained_layout=True)
axes[0].hist(finite_ndwi, bins=20, range=(-1, 1))
axes[0].axvline(cutoff, color="red", label=f"Otsu: {cutoff:.4f}")
axes[0].set(xlabel="NDWI", ylabel="Pixel count")
axes[0].legend()
image = axes[1].imshow(water_mask, vmin=0, vmax=1, cmap="Blues")
axes[1].set(title="Water mask", xlabel="Column", ylabel="Row")
figure.colorbar(image, ax=axes[1], ticks=[0, 1], label="0: non-water / 1: water")
plt.show()
water_pixels = int(np.count_nonzero(water_mask))
print(f"Observed: {water_pixels} water pixels out of {water_mask.size}")
print(f"Your prediction: {predicted_water_pixels}")
if predicted_water_pixels is not None:
    print(
        "Prediction matches!"
        if predicted_water_pixels == water_pixels
        else "Revisit the crop and threshold rule."
    )

## 6. Inspect the STAC result

Package the mask and validate its metadata through PySTAC, including declared extensions. Which information makes this raster useful outside the notebook?

In [ ]:
stac_impl.execute(input_item=source_catalog, water_body=Path("otsu.tif"))
result_item = read_item(Path("catalog"))
validate_item(result_item)
print("STAC validation passed")
print(f"Item: {result_item.id}")
print(f"Acquired: {result_item.datetime}")
print(f"Bounds: {result_item.bbox}")
print(f"Mask: {result_item.assets['data'].get_absolute_href()}")

## 7. Experiment

Predict first, then change section 2 and rerun sections 3–6:

- Try AOI `1,1,4,7`, then `4,1,7,7`. What happens when every pixel equals the threshold? A strict greater-than rule does not guarantee water in a uniform scene.
- Reverse `bands` to `["nir", "green"]`. Compare NDWI signs and the location of water, even if the count stays the same.

The next cell deliberately uses unequal grids in a separate directory. Explain the validation error.

In [ ]:
experiment_directory = run_directory / "mismatched-grids"
experiment_directory.mkdir(exist_ok=True)
os.chdir(experiment_directory)
try:
    crop_impl.execute(input_item=source_catalog, aoi="1,1,7,7", epsg="4326", band="green")
    crop_impl.execute(input_item=source_catalog, aoi="2,2,7,7", epsg="4326", band="nir")
    try:
        norm_diff_impl.execute(rasters=[Path("crop_green.tif"), Path("crop_nir.tif")])
    except ValueError as error:
        print(f"Expected validation error: {error}")
    else:
        raise RuntimeError("Expected unequal grids to be rejected.")
finally:
    os.chdir(run_directory)

## Reflect

1. Why does band order belong in the workflow contract?
2. Why does cropping to the same AOI not guarantee grid alignment?
3. What does this controlled fixture verify? Does it measure real-scene accuracy?

Compare [the solution](solution.md) after finishing the default experiment. Outputs remain in the printed directory. You may delete that run directory after closing the notebook.